In [1]:
import pandas as pd

df = pd.read_csv("data/raw/maintenance.csv")
print(df["Equipment_ID"].unique())       # all asset tags already in P-101 format
print(df["OrderType"].value_counts())    # CM, PM etc.
print(len(df))                           # total rows


C:\Users\ASUS\AppData\Local\Temp\ipykernel_27060\775515602.py:3: DtypeWarning: Columns (0: WorkOrder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/raw/maintenance.csv")


<StringArray>
[     'P-101',      'P-102',          nan,      'P-103',      'P-201',
      'P-202',       'P-22',      'P-301',      'P-302',     'PP-302',
 ...
     'dp-017',     'gs-018',     'SC0-18', '  HP-018  ',     'hp-018',
       'P119',      'p-219',     'R-0188',     'TK0-22',    'Blr-019']
Length: 48701, dtype: str
OrderType
CM         545836
PM         426861
PDM         45814
  CM         4590
  PM         3609
-            3030
  PDM         364
pdm           119
DPM           118
PMD           111
PPDM           89
PDDM           86
PDMM           86
Name: count, dtype: int64
1048779


In [3]:
import pandas as pd

df = pd.read_csv("data/raw/maintenance.csv", low_memory=False)

# 1. Normalize
df["Equipment_ID"] = df["Equipment_ID"].astype(str).str.strip().str.upper()
df["OrderType"]    = df["OrderType"].astype(str).str.strip().str.upper()

# 2. Filter valid tags — str.match with na=False skips NaN/pd.NA safely
df = df[df["Equipment_ID"].str.match(
    r'^(P|PP|COMP|C|HX|E|V|TK|T|MTR|M|B|BLR|R)-\d+[A-Z]?$',
    na=False
)]

# 3. Normalize OrderType
ORDER_MAP = {
    "CM": "CM", "PM": "PM", "PDM": "PDM",
    "DPM": "PDM", "PMD": "PM", "PPDM": "PDM",
    "PDDM": "PDM", "PDMM": "PDM",
}
df["OrderType"] = df["OrderType"].map(ORDER_MAP)
df = df[df["OrderType"].notna()]

# 4. See what clean assets remain
print("Clean Equipment_IDs:")
print(df["Equipment_ID"].value_counts().head(20))
print(f"\nTotal clean rows: {len(df)}")


Clean Equipment_IDs:
Equipment_ID
P-601    314
P-502    314
P-504    313
P-515    313
P-512    311
P-597    309
P-506    308
P-523    308
P-508    307
P-510    307
P-511    307
P-517    306
P-501    304
P-518    304
P-534    304
P-564    304
P-587    304
P-514    303
P-547    303
P-559    303
Name: count, dtype: int64

Total clean rows: 410411


In [4]:
# Check equipment type diversity in clean data
print("All prefix types found:")
prefix = df["Equipment_ID"].str.extract(r'^([A-Z]+)-')
print(prefix[0].value_counts())

print("\n--- Are P-101, P-102, P-103 in clean data? ---")
for tag in ["P-101", "P-102", "P-103", "P-201", "P-202", "P-301"]:
    count = (df["Equipment_ID"] == tag).sum()
    print(f"  {tag}: {count} rows")

print(f"\nTotal unique clean Equipment_IDs: {df['Equipment_ID'].nunique()}")


All prefix types found:
0
P      167381
TK      68140
M       60633
HX      53107
R       37995
BLR     22690
PP        167
C         129
T          85
E          41
V          26
B          17
Name: count, dtype: int64

--- Are P-101, P-102, P-103 in clean data? ---
  P-101: 45 rows
  P-102: 89 rows
  P-103: 133 rows
  P-201: 145 rows
  P-202: 191 rows
  P-301: 206 rows

Total unique clean Equipment_IDs: 6388


In [7]:
import pandas as pd

# --- Manually chosen pumps ---
PUMP_TAGS = ["P-101", "P-102", "P-201"]

# --- Auto-pick most frequent tag from each other type ---
other_types = ["TK", "M", "HX", "R", "BLR", "C"]
prefix = df["Equipment_ID"].str.extract(r'^([A-Z]+)-')[0]

auto_tags = []
for ptype in other_types:
    mask = prefix == ptype
    if mask.sum() == 0:
        continue
    top_tag = df[mask]["Equipment_ID"].value_counts().idxmax()
    count   = df[mask]["Equipment_ID"].value_counts().max()
    auto_tags.append(top_tag)
    print(f"{ptype}: selected {top_tag} ({count} rows)")

SELECTED_TAGS = PUMP_TAGS + auto_tags
print(f"\nFinal selected assets: {SELECTED_TAGS}")

# --- Sample 60 rows per asset (simple loop, no groupby.apply) ---
frames = []
for tag in SELECTED_TAGS:
    subset = df[df["Equipment_ID"] == tag]
    frames.append(subset.sample(min(60, len(subset)), random_state=42))
df_demo = pd.concat(frames, ignore_index=True)

# --- Keep only RAG-useful text columns ---
TEXT_COLS = [
    "Equipment_ID", "OrderType",
    "WorkOrderDescription", "OperationDescription",
    "Maintenance_activity_type", "CreatedDate",
    "ActualStartDate", "ActualFinishDate",
    "OrderStatus", "MaintenanceWorkCenter"
]
keep = [c for c in TEXT_COLS if c in df_demo.columns]
df_demo = df_demo[keep]

print(f"\nRows per asset:")
print(df_demo["Equipment_ID"].value_counts())
print(f"\nTotal rows: {len(df_demo)}")

df_demo.to_csv("data/documents/work_orders_clean.csv", index=False)
print("\nSaved to data/documents/work_orders_clean.csv")


TK: selected TK-482 (144 rows)
M: selected M-017 (165 rows)
HX: selected HX-305 (122 rows)
R: selected R-201 (84 rows)
BLR: selected BLR-118 (56 rows)
C: selected C-369 (4 rows)

Final selected assets: ['P-101', 'P-102', 'P-201', 'TK-482', 'M-017', 'HX-305', 'R-201', 'BLR-118', 'C-369']

Rows per asset:
Equipment_ID
P-102      60
P-201      60
TK-482     60
M-017      60
HX-305     60
R-201      60
BLR-118    56
P-101      45
C-369       4
Name: count, dtype: int64

Total rows: 465

Saved to data/documents/work_orders_clean.csv


In [2]:
!pip install fpdf2



Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


SyntaxError: invalid syntax (379249682.py, line 1)